# Informativity exploratory tests

This file is mostly for testing the informativity Data-Driven Control (DDC) in the literature.

In [1]:
# Default packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shutil
import scipy
from scipy.constants import c, pi
from scipy import signal
from matplotlib import cm
from numpy.fft import fft, fftshift

# For nice plots
import plotly.io as pio
from plotly import subplots
import plotly.express as px
import plotly.graph_objects as go

# --- Dark/Interactive Theme ---
theme = "plotly_dark"

layout_defaults = dict(
    template=theme,
    paper_bgcolor="rgba(0,0,0,0)", # Transparent background
    plot_bgcolor="rgba(0,0,0,0)",  # Transparent plot area
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor="rgba(0, 0, 0, 0.4)",
        bordercolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.12)"),
    yaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.12)"),
    width=650,
    height=325,
    autosize=False, # Prevents the browser from stretching it
    margin=dict(l=40, r=40, t=40, b=40),
    title = dict(
        x=0.5,
        xanchor='center'
    ),
    font = dict(
        family = "Helvetica",
        size = 14
    )
)

# Apply the defaults globally
pio.templates.default = theme
pio.templates[theme].layout.update(layout_defaults); # type: ignore

# For inline plots
%matplotlib inline

# This is for latex in plotly - if it doesn't work comment it
from IPython.display import display, HTML
display(HTML("<script src='https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML'></script>"))

# Calc demonstration packages
import handcalcs.render
handcalcs.set_option("display_precision", 2) # type: ignore
import forallpeople as si
si.environment('default', top_level= True)

# Type defining for pylance
from typing import TYPE_CHECKING
# This block is only seen by Pylance/IDE
if TYPE_CHECKING:
    m = si.m                    # type: ignore
    kg = si.kg                  # type: ignore
    Ohm = si.Ohm                # type: ignore
    V = si.V                    # type: ignore
    A = si.A                    # type: ignore
    Hz = si.Hz                  # type: ignore
    W = si.W                    # type: ignore
    C = si.C                    # type: ignore
    F = si.F                    # type: ignore
    Wb = si.Wb                  # type: ignore
    T = si.T                    # type: ignore
    H = si.H                    # type: ignore
    Celsius = si.Celsius        # type: ignore
    lux = si.lux                # type: ignore
    s = si.s                    # type: ignore
    # Add any other units you use frequently

# Some unit naming conventions
ohm = Ohm

import cvxpy as cp
import control as ct

# Print all solvers available in your current environment
# Using clarabel for now
print(f"Listing installed solvers below:")
print(cp.installed_solvers())

# plt.rcParams.update(
#     {
#         "text.usetex": True,
#         "font.family": "typeface",
#     }
# )  # Makes plotting slower but prettier (requires LaTeX installed)

# For debugging personal modules
# %load_ext autoreload
# %autoreload 2

Listing installed solvers below:
['MOSEK', 'CLARABEL', 'SCS', 'SCIPY', 'HIGHS', 'OSQP']


## Example 3.6 from Lidong Li's thesis

In [8]:
Atrue = np.array(
    [
        [0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0],
        [0, 0.0548, 0.9217, 0, 0, 0, 0],
        [0.4882, 0, 0, 0.9024, 0, 0, 0],
        [0, 0, 0, 0, 0.8607, 0, 0],
        [0, 0, 0, 0, 0, 0.8607, 0],
        [0, 0, 0, 0, 0, 0, 0.9024],
    ]
)

Btrue = np.array(
    [
        [1, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0.0348, 0, 0.2298],
        [0, 0.0808, 0],
        [0, 0.3564, 0],
    ]
)

n, m = Atrue.shape[0], Btrue.shape[1]
C = np.eye(n)

# Measurement and actuation noise
dx = np.sqrt(5e-5)
du = np.sqrt(5e-5)
N = 200 + 1 # T+1

# Input with measurement noise
Um = np.random.randn(m, N)

# Intantaneously bounded noise for input and states
raw_noise_u = np.random.normal(size=(m, N))
raw_noise_x = np.random.normal(size=(n, N))
norms_u = np.linalg.norm(raw_noise_u, axis=0, ord=2)
norms_x = np.linalg.norm(raw_noise_x, axis=0, ord=2)
unit_directions_u = raw_noise_u / norms_u
unit_directions_x = raw_noise_x / norms_x
radii_u = du * (np.random.rand(1, N))**(1/m)
radii_x = dx * (np.random.rand(1, N))**(1/n)
n_u = unit_directions_u * radii_u  # Shape: (m, N), satisfying ||w(k)||_2^2 <= epsilon
n_x = unit_directions_x * radii_u  # Shape: (n, N), satisfying ||w(k)||_2^2 <= epsilon

# Applies noise to input
U = Um - n_u

# Runs sim and gathers data
sys = ct.ss(Atrue, Btrue, C, 0, dt=1.0)
t_sim, X = ct.forced_response(sys, U=U, initial_state=np.random.randn(n)) # type: ignore

# Adds measurement noise to states
Xm = X + n_x

In [9]:
# Mounts data matrices
Thor = N-1
Tinit = 0
X0 = Xm[:, Tinit:Thor+Tinit]  # X(k)
X1 = Xm[:, Tinit+1:Thor+Tinit+1]  # X(k+1)
U0 = Um[:, Tinit:Thor+Tinit]  # U(k)

# Mounts LMI matrices
Theta = Thor*(2*dx**2 + du**2)*np.eye(2*n+m)
Theta11 = Theta[:n, :n]
Theta12 = Theta[:n, n:]
Theta21 = Theta12.T
Theta22 = Theta[n:, n:]
aux1 = np.block(
    [
        [X0],
        [U0]
    ]
)
Acal = aux1@aux1.T - Theta22
Bcal = -X1@aux1.T + Theta12
Ccal = X1@X1.T - Theta11

# Mounts LMI
# Variables
P = cp.Variable((n, n), PSD=True)
Y = cp.Variable((m, n))

# Block 1
lmi_m1 = cp.bmat(
    [
        [-P - Ccal, np.zeros((n, n)), Bcal],
        [np.zeros((n, n)), -P, cp.hstack([P, Y.T])],
        [Bcal.T, cp.vstack([P,Y]), -Acal] # type: ignore
    ]
)

# Full LMI
tol = 1e-12
constraints = [
    lmi_m1 << 0
]
prob = cp.Problem(cp.Minimize(1), constraints) # type: ignore
prob.solve(solver = 'MOSEK', verbose=False)
# Extracts K and checks for stability
if(prob.status == 'optimal'):
    print('Success?')
    Kfound = Y.value @ np.linalg.inv(P.value) # type: ignore
    print(Kfound)

In [10]:
# Now tries to solve with inst. noise bounds
theta = 2*dx**2 + du**2
# Variables
P = cp.Variable((n, n), PSD=True)
Y = cp.Variable((m, n))
tau = cp.Variable(Thor, nonneg=True)  # tau_0, ..., tau_{T-1} >= 0

# Block 1
lmi_m1 = cp.bmat(
    [
        [-P, np.zeros((n, n)), np.zeros((n, m)), np.zeros((n, n))],
        [np.zeros((n, n)), P, Y.T, np.zeros((n, n))],
        [np.zeros((m, n)), Y, np.zeros((m, m)), Y],
        [np.zeros((n, n)), np.zeros((n, n)), Y.T, -P]
    ]
)

dim_total = 3 * n + m
aux2 = np.zeros((dim_total, dim_total))
aux2[:2*n+m, :2*n+m] = theta*np.eye(2*n+m)
lmi_m2 = []

for k in range(Thor):
    # Construct column vector v_k:
    # [ x_m(k+1) ] (n x 1)
    # [ -x_m(k)  ] (n x 1)
    # [ -u_m(k)  ] (m x 1)
    # [    0     ] (m x 1)
    v_k = np.block([
        Xm[:, k + 1],
        -Xm[:, k],
        -Um[:, k],
        np.zeros(n)
    ]).reshape(-1, 1)  # Column vector of size (2n + 2m, 1)
    
    # Accumulate scaled outer products (v_k @ v_k.T)
    # tau[k] scales this numeric matrix inside CVXPY
    lmi_m2.append(tau[k] * (v_k @ v_k.T - aux2))

lmi_m3 = sum(lmi_m2)
# Full LMI
tol = 1e-12
constraints = [
    lmi_m1 - lmi_m3 << 0
]
prob = cp.Problem(cp.Minimize(1), constraints) # type: ignore
prob.solve(solver = 'MOSEK', verbose=False)
if(prob.status == 'optimal'):
    print('Success?')
    Kfound = Y.value @ np.linalg.inv(P.value) # type: ignore
    print(Kfound)

Success?
[[-0.20510563 -0.10442135 -1.0994341  -0.25943878  0.10902216 -2.28377506
   0.57464876]
 [ 0.04702041  0.02638725  1.0178035  -0.11432121  0.02041257  5.53718414
  -3.17922796]
 [-0.04748868  0.23255725  5.51914813 -0.46974692 -2.2515736  13.53328294
  -3.1815233 ]]


In [18]:
# Plotting sim data
fig = subplots.make_subplots(rows=3)
fig.add_trace(
    go.Scattergl(
        y=Um[0,:],
        mode="lines",
        name=r"$U(k)$",
    ),
    row=1,
    col=1
)
for state in range(n):
    fig.add_trace(
        go.Scattergl(
            y=Xm[state,:],
            mode="lines",
            name=f"$X(k)$",
            legendgroup="states",
            showlegend=(state == 0)
        ),
    row=2,
    col=1
    )


fig.add_trace(
    go.Scattergl(
        y=np.linalg.norm(n_u, axis=0, ord=2),
        mode="lines",
        name="$\|n_x(k)\|_2$"
    ),
    row=3,
    col=1
)

# Update axis labels and layout
#fig.update_yaxes(title_text="Voltage (V)", row=1)
#fig.update_yaxes(title_text="Voltage (V)", row=2)
#fig.update_xaxes(title_text="Time (s)", row=2)

fig.update_layout(
    height = 400
)

fig.show()

## Example 6.3.2 - van Waarden's book

Right now, the code below only agrees qualitatively with the book. Numerically, there are significant differences, mainly the fact that, for $\epsilon = 0.5$ the solver only returns a 'good' result in 68/100 cases, whereas the reported result is 100/100.

In [ ]:
# Testing packages with example 6.3.2 from van Waarde's book
#np.random.seed(67)  # For reproducibility
success_count = 0
stab_count = 0
N_exp = 100

# Running 100 times
for i in range(N_exp):
    # 1. Generate LTI System Data (python-control)
    Atrue = np.array([[0.850, -0.038, -0.38], 
                    [0.735, 0.815, 1.594],
                    [-0.664, 0.697, -0.064]])
    Btrue = np.array([[1.431, 0.705], 
                    [1.620, -1.129],
                    [0.913, 0.369]])

    n, m = Atrue.shape[0], Btrue.shape[1]

    # 2. Augment Input Matrix B_aug = [B, I_n] so noise w(k) enters directly as input
    Baug = np.hstack([Btrue, np.eye(n)])
    C = np.eye(n)

    # Simulate trajectory with input sequence U
    N = 21 # >= T+1
    U = np.random.randn(m, N)

    # Generate uniform direction vectors (Gaussian normalized)
    epsilon = 2.2
    raw_noise = np.random.normal(size=(n, N))
    norms = np.linalg.norm(raw_noise, axis=0, ord=2)
    unit_directions = raw_noise / norms

    # === NOTE === #
    # The way the sampling is done below doesn't correspon with the results in the book
    # Sample random radii up to sqrt(epsilon)
    radii = np.sqrt(epsilon) * (np.random.rand(1, N))**(1/n)
    W = unit_directions * radii  # Shape: (n, N), satisfying ||w(k)||_2^2 <= epsilon

    # This only almost works for epsilon = 0.5
    W = np.sqrt(epsilon)*np.random.randn(n, N)

    # Combine inputs: [u(k); w(k)] -> Shape: (m + n, N)
    U_augmented = np.vstack([U, W])

    # Runs sim and gathers data
    sys = ct.ss(Atrue, Baug, C, 0, dt=1.0)
    t_sim, X = ct.forced_response(sys, U=U_augmented, initial_state=np.random.randn(n)) # type: ignore

    Thor = 20
    Tinit = 0
    Xminus = X[:, Tinit:Thor+Tinit]  # X(k)
    Xplus  = X[:, Tinit+1:Thor+Tinit+1]  # X(k+1)
    Uminus = U[:, Tinit:Thor+Tinit]  # U(k)

    # 2. Informativity / LMI Formulation in CVXPY

    # Definition of Phi
    Phi_noise_bound = epsilon
    Phi11 = Thor*Phi_noise_bound*np.eye(n)
    Phi12 = np.zeros((n, Thor))
    Phi21 = Phi12.T
    Phi22 = -np.eye(Thor)

    # Phi
    Phi = np.block([[Phi11, Phi12], [Phi21, Phi22]])

    # Variables
    P = cp.Variable((n, n), PSD=True)
    L = cp.Variable((m, n))
    beta = cp.Variable(pos = True)

    # Theorem 6.25
    # Block 1
    lmi_m1 = cp.bmat(
        [
            [P - beta * np.eye(n), np.zeros((n, n)), np.zeros((n, m)), np.zeros((n, n))],
            [np.zeros((n, n)), -P, -L.T, np.zeros((n, n))],
            [np.zeros((m, n)), -L, np.zeros((m, m)), L],
            [np.zeros((n, n)), np.zeros((n, n)), L.T, P],
        ]
    )

    # Block 2
    lmi_data = np.block(
        [
            [np.eye(n), Xplus],
            [np.zeros((n, n)), -Xminus],
            [np.zeros((m, n)), -Uminus],
            [np.zeros((n, n)), np.zeros((n, Thor))],
        ]
    )

    # Block 2
    lmi_m2 = lmi_data @ Phi @ lmi_data.T

    # Full LMI
    tol = 1e-12
    constraints = [
        lmi_m1 - lmi_m2 >> 0,
        P >> tol*np.eye(n),
        beta >= tol
    ]
    prob = cp.Problem(cp.Minimize(1), constraints)
    # The clarabel options clearly need more refining
    # if mosek is an option, consider using it
    clarabel_opts = {
        'max_iter': int(1e5),
        'tol_gap_abs':1e-6,
        'tol_gap_rel':1e-6,
        'tol_feas':1e-8,
        'max_step_fraction':0.2,
        'iterative_refinement_max_iter': 20,
    }
    prob.solve(solver = 'MOSEK', verbose=False)

    # Extracts K and checks for stability
    if(prob.status == 'optimal'):
        success_count+=1

print(f"Success rate = {success_count/N_exp*100.0}%")

Success rate = 40.0%


## Example 8.2.1 from van Waarden's book

Checking to see if it yields the same result as the book

Issues:
    - Mosek doesn't run with default configs
    - SCS does run and returns a similar cost, though it's imprecise


In [51]:
# Example 8.2.1 - van Waarden

# System's matrices
Atrue = np.array(
    [
        [1.000, -0.374, -0.190, -0.321, 0.056, -0.026],
        [0.000, 0.982, 0.010, -0.000, -0.003, 0.001],
        [0.000, 0.115, 0.975, -0.000, -0.269, 0.191],
        [0.000, 0.001, 0.010, 1.000, -0.001, 0.001],
        [0.000, 0.000, 0.000, 0.000, 0.741, 0.000],
        [0.000, 0.000, 0.000, 0.000, 0.000, 0.741],
    ]
)

# B_true (2x6 transposed to 6x2 matrix)
Btrue = np.array(
    [
        [0.007, 0.000, -0.043, 0.000, 0.259, 0.000],
        [-0.003, 0.000, 0.030, 0.000, 0.000, 0.259],
    ]
).T

A_cont = np.array([
    [-0.5,  0.2, -0.1,  0.0,  0.0,  0.1],
    [-0.3, -0.8,  0.4, -0.2,  0.0,  0.0],
    [ 0.0, -0.2, -0.6,  0.3, -0.1,  0.0],
    [ 0.1,  0.0, -0.4, -0.9,  0.2, -0.1],
    [ 0.0,  0.1,  0.0, -0.3, -0.7,  0.4],
    [-0.2,  0.0,  0.1,  0.0, -0.5, -1.0]
])
# Matrix exponential ensures discrete-time matrix A is strictly stable
Atrue = scipy.linalg.expm(A_cont * 0.2) 

# Verify spectral radius < 1
eigvals = np.abs(np.linalg.eigvals(Atrue))
print(f"Max eigenvalue magnitude of A: {np.max(eigvals):.4f}")

# 2. Input Matrix (6x2)
Btrue = np.array([
    [ 0.10, -0.05],
    [ 0.20,  0.15],
    [-0.10,  0.30],
    [ 0.05, -0.20],
    [ 0.25,  0.00],
    [ 0.00,  0.18]
])

n, m = Atrue.shape[0], Btrue.shape[1]

Ctrue = np.eye(n)
Dtrue = 0.0

# Cost metric matrices
Ccost = np.array([[0, 0, 0, 0, 0, 1]])
Dcost = 0.0

# Dimension of z
nzcost = Ccost.shape[0]

# Augment Input Matrix B_aug = [B, I_n] so noise w(k) enters directly as input
Baug = np.hstack([Btrue, np.eye(n)])

# Sim. parameters
N = 751
sigma_w = 0.005

# Generates some signals for the sim

# Input
U = np.random.randn(m, N)

# Noise
W = sigma_w*np.random.randn(n, N)

# Combine inputs: [u(k); w(k)] -> Shape: (m + n, N)
Uaug = np.vstack([U, W])

# Runs sim and gathers data
sys = ct.ss(Atrue, Baug, Ctrue, 0, dt=1.0)
tsim, X = ct.forced_response(sys, U=Uaug, initial_state=np.random.randn(n)) # type: ignore

Thor = 750
Tinit = 0
Xminus = X[:, Tinit:Thor+Tinit].copy()  # X(k)
Xplus  = X[:, Tinit+1:Thor+Tinit+1].copy()  # X(k+1)
Uminus = U[:, Tinit:Thor+Tinit].copy()  # U(k)

# 2. Informativity / LMI Formulation in CVXPY

# Definition of Phi
Phi_noise_bound = 1.35*sigma_w**2
Phi11 = Thor*Phi_noise_bound*np.eye(n)
Phi12 = np.zeros((n, Thor))
Phi21 = Phi12.T
Phi22 = -np.eye(Thor)

# Phi
Phi = np.block([[Phi11, Phi12], [Phi21, Phi22]])

# Mounts the LMIs in 8.30
# Variables
Y = cp.Variable((n, n), PSD=True)
Z = cp.Variable((n, n), symmetric=True)
L = cp.Variable((m, n))
alpha = cp.Variable(nonneg=True)
beta = cp.Variable(pos=True)
gamma = cp.Variable(pos=True)

# Matrix blocks
lmi_m1 = cp.bmat(
    [
        [Y - beta * np.eye(n), np.zeros((n, n)), np.zeros((n, m)), np.zeros((n, n)), np.zeros((n, nzcost))],
        [np.zeros((n, n)), np.zeros((n, n)), np.zeros((n, m)), Y, np.zeros((n, nzcost))],
        [np.zeros((m, n)), np.zeros((m, n)), np.zeros((m, m)), L, np.zeros((m, nzcost))],
        [np.zeros((n, n)), Y, L.T, Y, (Ccost@Y).T],
        [np.zeros((nzcost, n)), np.zeros((nzcost, n)), np.zeros((nzcost, m)), Ccost@Y, np.eye(nzcost)]
    ]
)

lmi_data = np.block(
    [
        [np.eye(n), Xplus],
        [np.zeros((n, n)), -Xminus],
        [np.zeros((m, n)), -Uminus],
        [np.zeros((n, n)), np.zeros((n, Thor))],
        [np.zeros((nzcost, n)), np.zeros((nzcost, Thor))]
    ]
)

lmi_m2 = lmi_data @ Phi @ lmi_data.T

lmi_m3 = cp.bmat(
    [
        [Y, (Ccost@Y).T],
        [Ccost@Y, np.eye(nzcost)]
    ]
)

lmi_m4 = cp.bmat(
    [
        [Z, np.eye(n)],
        [np.eye(n), Y]
    ]
)

# Full LMI
tol = 1e-6
constraints = [
    Y >> tol*np.eye(n),
    Y << 1/tol * np.eye(n),
    beta >= tol,
    gamma >= tol,
    lmi_m1 - alpha*lmi_m2 >> 0, # 1
    lmi_m3 >> tol*np.eye(n+nzcost), # 2
    lmi_m4 >> 0, # 3
    cp.trace(Z) <= gamma
]

# Define MOSEK specific tuning parameters
mosek_params = {
    'MSK_DPAR_OPTIMIZER_MAX_TIME': 100.0,
    'MSK_IPAR_INTPNT_SOLVE_FORM': 'MSK_SOLVE_DUAL',
    'MSK_DPAR_INTPNT_CO_TOL_PFEAS': 1e-6,
    'MSK_DPAR_INTPNT_CO_TOL_DFEAS': 1e-6,
    'MSK_DPAR_INTPNT_CO_TOL_REL_GAP': 1e-5,
}

prob = cp.Problem(cp.Minimize(gamma), constraints)
prob.solve(solver = 'MOSEK', verbose=True, mosek_params=mosek_params)
print(f"Status = {prob.status}")

(CVXPY) Aug 05 10:34:32 AM: Your problem has 87 variables, 709 constraints, and 0 parameters.
(CVXPY) Aug 05 10:34:32 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 05 10:34:32 AM: DCP verification time: 0.0000 seconds.
(CVXPY) Aug 05 10:34:32 AM: Expression tree has 9 nodes.
(CVXPY) Aug 05 10:34:32 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 05 10:34:32 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 05 10:34:32 AM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Aug 05 10:34:32 AM: Compiling problem (target solver=MOSEK).
(CVXPY) Aug 05 10:34:32 AM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ExactCone2Cone -> EliminateZeroSized -> ConeMatrixStuffing -> MOSEK
(CVXPY) Aug 05 10:34:32 AM: Applying reduction Dcp2Cone
(CVXPY) Aug 05 10:34:32 AM: Applying reduction CvxAttr2Constr
(CVXPY) Aug 05 1

Max eigenvalue magnitude of A: 0.8812
                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------




(CVXPY) Aug 05 10:34:32 AM: 30  7.1e-15  9.7e-10  2.5e-07  -2.35e-01  5.736226494e+04   5.553550332e+04   9.7e-16  0.10  
(CVXPY) Aug 05 10:34:32 AM: Optimizer terminated. Time: 0.11    
(CVXPY) Aug 05 10:34:32 AM: 
(CVXPY) Aug 05 10:34:32 AM: 
(CVXPY) Aug 05 10:34:32 AM: Interior-point solution summary
(CVXPY) Aug 05 10:34:32 AM:   Problem status  : UNKNOWN
(CVXPY) Aug 05 10:34:32 AM:   Solution status : UNKNOWN
(CVXPY) Aug 05 10:34:32 AM:   Primal.  obj: 5.7362264941e+04    nrm: 2e+10    Viol.  con: 4e-03    var: 0e+00    barvar: 0e+00  
(CVXPY) Aug 05 10:34:32 AM:   Dual.    obj: 5.5535503317e+04    nrm: 1e+06    Viol.  con: 0e+00    var: 1e-06    barvar: 1e+00  


SolverError: Solver 'MOSEK' failed. Try another solver, or solve with verbose=True for more information.

In [199]:
# debug

# Plotting sim data
fig = subplots.make_subplots(rows=3)
fig.add_trace(
    go.Scatter(
        y=U[0,:],
        mode="lines",
        name=r"$U(k)$",
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        y=X[5,:],
        mode="lines",
        name=r"$X(0)$",
    ),
    row=2,
    col=1
)
fig.add_trace(
    go.Scatter(
        y=np.linalg.norm(W, axis=0, ord=2),
        mode="lines",
    ),
    row=3,
    col=1
)

# Update axis labels and layout
#fig.update_yaxes(title_text="Voltage (V)", row=1)
#fig.update_yaxes(title_text="Voltage (V)", row=2)
#fig.update_xaxes(title_text="Time (s)", row=2)

fig.update_layout(
    height = 400
)

fig.show()
print(np.max((np.linalg.norm(W, axis=0, ord=2))**2))

0.000515050545212438


In [40]:
px.line(Xplus.T)
